# Public-source note

Final S5e enhancer-state + RNA-defined promoter/TSS motif heatmap notebook. Outputs stripped and paths sanitized.

In [ ]:

library(circlize)
library(dplyr)
library(ggplot2)
library(data.table)
library(GenomicRanges)
library(BiocParallel)
library(ComplexHeatmap)

In [ ]:
Enh_figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4/figures_histone_chromhmm/"
TSS_figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4/figures_clustered_TSS/"
figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4_V2/"

In [ ]:
# pattren_names_df <- read.table(
#   paste(Enh_figure_dir,"/convert_names_pattren_names_counts.csv",sep=""),
#   header = TRUE,
#   sep = ",",
#     row.names = 1,
#   stringsAsFactors = FALSE
# )


In [ ]:
pattren_names_df <- read.table(
  paste("5e_EP_motifs_min_prop_0.01_with_KC_names.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)


In [ ]:
nrow(pattren_names_df)

In [ ]:
head(pattren_names_df)

In [ ]:
length(unique(pattren_names_df$pattern))

In [ ]:
length(unique(pattren_names_df$pattern_num))

In [ ]:
length(unique(rownames(pattren_names_df)))

In [ ]:
histone_scaled_df <- read.table(
  paste(Enh_figure_dir,"/Finemo_merged_Pr_Histone_sum_importance_over_state_size_sep_counts.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)




In [ ]:
nrow(histone_scaled_df)

In [ ]:
head(histone_scaled_df)

In [ ]:
histone_motif_prop_df <- read.table(
  paste(Enh_figure_dir,"/Finemo_merged_Pr_histone_motif_prop_counts.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)


In [ ]:
head(histone_motif_prop_df)

In [ ]:
histone_motif_prop_with_names_df <- merge(pattren_names_df, histone_motif_prop_df, by = "pattern")
head(histone_motif_prop_with_names_df)

In [ ]:
nrow(histone_motif_prop_with_names_df)

In [ ]:
# min_prop <- 0.05

# motif_keep <- histone_motif_prop_with_names_df %>%
#     filter(prop >= min_prop) %>%
#     distinct(all_matches) %>%
#     pull(all_matches)


In [ ]:
TSS_scaled_df <- read.table(
  paste(TSS_figure_dir,"/Finemo_merged_sum_imp_per_size_cleaned_Starting_cell_types_clusters_counts.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)


In [ ]:
nrow(TSS_scaled_df)

In [ ]:
head(TSS_scaled_df)

In [ ]:
TSS_motif_max_frac_df <- read.table(
  paste(TSS_figure_dir,"/TSS_cluster_motif_max_frac_counts.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)


In [ ]:
# min_prop <- 0.05

# motif_keep <- TSS_motif_max_frac_df %>%
#     filter(max_frac_cebp_with_motif >= min_prop) %>%
#     distinct(all_matches) %>%
#     pull(all_matches)


In [ ]:
# length(motif_keep)

In [ ]:
tss_sub_cols <- c('RNA_cluster_1_MEF',
                  'RNA_cluster_2_NSC',
                  'RNA_cluster_3_ESC',
                  'RNA_cluster_4_Bcell_Graf'
                 )

In [ ]:
nrow(histone_scaled_df)


In [ ]:
histone_TSS_scaled_df <- merge(histone_scaled_df, TSS_scaled_df[,tss_sub_cols], by = "row.names")
rownames(histone_TSS_scaled_df) <- histone_TSS_scaled_df$Row.names
histone_TSS_scaled_df$Row.names <- NULL

In [ ]:
head(histone_TSS_scaled_df)

In [ ]:
nrow(histone_TSS_scaled_df)

In [ ]:
program_order2 <- c(
  "MEF",
  'NSC' ,
  'ESC' ,
'Bcell_Graf' 
)

histone_state_cols <-paste0("H",c(
	
	4 , #MEF Enh
    5 , #NSC Enh
    6,7 , #ESC Enh
    8  # Bcell Enh
	))

histone_state_sub_cols <-paste0("H",c(
	
	4 , #MEF Enh
    5 , #NSC Enh
    6 , #ESC Enh
    8  # Bcell Enh
	# 9 , #MEF NSC Enh
	# 10,11  #MEF NSC ESC Enh
	))


histone_states_split_vec <- c(

		rep("MEF\nEnh",1),
        rep("NSC\nEnh",1),
            rep("ESC\nEnh",2),
		rep("Bcell\nEnh",1)
		)

histone_states_split_vec <- trimws(histone_states_split_vec)  # removes trailing spaces like "B " or "H "
histone_states_split_fac <- factor(histone_states_split_vec, levels = unique(histone_states_split_vec))  # keep first seen order

In [ ]:
histone_state_sub_cols

In [ ]:


program_split_vec <- c(
		rep("MEF",1),
        rep("NSC",1),
		rep("ESC",1),
		rep("Bcell",1)
		)

program_split_vec <- trimws(program_split_vec)  # removes trailing spaces like "B " or "H "
program_split_fac <- factor(program_split_vec, levels = unique(program_split_vec))  # keep first seen order

In [ ]:


TSS_split_vec <- c(
		rep("MEF\nTSS",1),
        rep("NSC\nTSS",1),
		rep("ESC\nTSS",1),
		rep("Bcell\nTSS",1)
		)

TSS_split_vec <- trimws(TSS_split_vec)  # removes trailing spaces like "B " or "H "
TSS_split_fac <- factor(TSS_split_vec, levels = unique(TSS_split_vec))  # keep first seen order

In [ ]:
# min_prop <- 0.01

# motif_keep <- histone_motif_prop_with_names_df %>%
#     filter(prop >= min_prop) %>%
#     distinct(all_matches) %>%
#     pull(all_matches)


In [ ]:
motif_keep <- pattren_names_df$pattern_num

In [ ]:
length(motif_keep)

In [ ]:
head(histone_scaled_df)

In [ ]:
nrow(histone_scaled_df)

In [ ]:
histone_scaled_df$rowname <- rownames(histone_scaled_df)
pattren_names_df$rowname <- rownames(pattren_names_df)
# sub_histone_scaled_sorted <- merge(histone_scaled_df, pattren_names_df, by = "rowname")

sub_histone_scaled_sorted <- merge(
  pattren_names_df,
  histone_scaled_df,
  by = "rowname",
  sort = FALSE
)

In [ ]:
head(sub_histone_scaled_sorted)

In [ ]:
nrow(sub_histone_scaled_sorted)

In [ ]:

# sub_histone_scaled=histone_scaled_df[motif_keep,]
# sub_histone_scaled$max_Pr <-colnames(sub_histone_scaled[,histone_state_sub_cols])[max.col(sub_histone_scaled[,histone_state_sub_cols], ties.method = "first")]



# sub_histone_scaled$max_Pr <- factor(sub_histone_scaled$max_Pr, levels = histone_state_sub_cols)

# sub_histone_scaled <- sub_histone_scaled[order(sub_histone_scaled$max_Pr), ]
# sub_histone_scaled$cluster <- as.numeric(sub_histone_scaled$max_Pr)
# sub_histone_scaled$rowname <- rownames(sub_histone_scaled)

In [ ]:
# unique(sub_histone_scaled['max_Pr'])

In [ ]:
nrow(sub_histone_scaled_sorted)

In [ ]:
sub_histone_scaled_sorted

In [ ]:
program_state_order_dict <- c(
  "H4"=c('H4'), 
  'H5' =c('H5'), 
  'H6' =c('H6'), 
'H8' =c('H8')
)

In [ ]:
# library(dplyr)

# sort_by_map_desc_keep_all <- function(df, group_col = "max_Pr", sort_map) {

#   out <- list()

#   # 1) sorted groups, in the order of sort_map keys
#   for (k in names(sort_map)) {

#     g <- df %>% filter(.data[[group_col]] == k)
#     if (nrow(g) == 0) next

#     col_to_sort <- sort_map[[k]]

#     # if multiple columns provided, sort by all of them
#     if (length(col_to_sort) > 1) {
#       cols <- col_to_sort[col_to_sort %in% colnames(df)]
#       if (length(cols) > 0) {
#         g <- g %>% arrange(!!!lapply(cols, function(x) desc(.data[[x]])))
#       }
#     } else {
#       # single column
#       if (col_to_sort %in% colnames(df)) {
#         g <- g %>% arrange(desc(.data[[col_to_sort]]))
#       }
#     }

#     out[[k]] <- g
#   }

#   sorted_part <- bind_rows(out)

#   # 2) append rows whose max_Pr is not in sort_map (unchanged)
#   leftovers <- df %>% filter(!as.character(.data[[group_col]]) %in% names(sort_map))

#   bind_rows(sorted_part, leftovers)
# }


# sub_histone_scaled_sorted <- sort_by_map_desc_keep_all(sub_histone_scaled, "max_Pr", program_state_order_dict)

In [ ]:
col_fun2 <- colorRamp2(
  c(-2, 0, 2),
  c("#3B4CC0", "#FFFFFF", "#E66101")
)

col_fun <- colorRamp2(
  c(-2, 0, 2),
  c("#3B4CC0", "#FFFFFF", "#B40426")
)


col_fun3 <- colorRamp2(
  c(-2, 0, 2),
  # c(-3, 0, 3),
  c("#3B4CC0", "#FFFFFF",  "red")
)

col_fun3 <- colorRamp2(
  c(-3, 0, 3),
  c("blue", "white", "red")
)

In [ ]:
outfile <- file.path( "Enh_TSS_Motif_plot.pdf")
pdf(outfile, width = 10, height = 30, useDingbats=FALSE)
# pdf(outfile, width = 10, height = 20, useDingbats=FALSE)
h1=ComplexHeatmap::Heatmap(histone_TSS_scaled_df[sub_histone_scaled_sorted$rowname,program_order2],
	cluster_columns=F, cluster_rows=F,cluster_column_slices = FALSE,
	#col = circlize::colorRamp2(c(min(mvh),max(mvh)), hcl_palette ="Tropic"),
	# col = circlize::colorRamp2(seq(0,1, length = 3), c("lightgrey", "white", "#3B6FB6")),
  col = col_fun3,
	row_split = sub_histone_scaled_sorted$cluster,border = TRUE,gap = unit(1, "mm"),
       column_split = program_split_fac,
	heatmap_legend_param = list(title = "Scaled((sum_importance))", border = "black", title_gp = gpar(col = "black", fontsize = 10)),
	width = unit(1, "null"))


h2=ComplexHeatmap::Heatmap(histone_TSS_scaled_df[sub_histone_scaled_sorted$rowname,histone_state_cols],
	cluster_columns=F, cluster_rows=F,cluster_column_slices = FALSE,
	# col = circlize::colorRamp2(seq(0,1, length = 3), c("lightgrey", "white", "#D1495B")),
  col = col_fun2,
	row_split = sub_histone_scaled_sorted$cluster,border = TRUE,gap = unit(1, "mm"),
	column_split = histone_states_split_fac,
	heatmap_legend_param = list(title = "Scaled((sum_importance_over_state_size))", border = "black", title_gp = gpar(col = "black", fontsize = 10)),
	width = unit(1.5, "null"))

h3=ComplexHeatmap::Heatmap(histone_TSS_scaled_df[sub_histone_scaled_sorted$rowname,tss_sub_cols],
	cluster_columns=F, cluster_rows=F,cluster_column_slices = FALSE,
	# col = circlize::colorRamp2(seq(0,1, length = 3), c("lightgrey", "white", "#D1495B")),
  col = col_fun,
	row_split = sub_histone_scaled_sorted$cluster,border = TRUE,gap = unit(1, "mm"),
	column_split = TSS_split_fac,
	heatmap_legend_param = list(title = "Scaled((sum_importance_over_state_size))", border = "black", title_gp = gpar(col = "black", fontsize = 10)),
	width = unit(1.5, "null"))


ht <- h2+h3
# ht <- h2 + h3

draw(ht, gap = unit(3, "mm"))
dev.off()
message("Saved: ", outfile)
print(ht)

In [ ]:
length(motif_keep)

In [ ]:
# manual_names_df <- read.table(
#   paste(figure_dir,"/Finemo_PrATAC_patterns_0.05_tomtom_manual_match.csv",sep=""),
#   header = TRUE,
#   sep = ",",
#     # row.names = FALSE,
#   stringsAsFactors = FALSE
# )
# head(manual_names_df)

In [ ]:
# nrow(manual_names_df)

In [ ]:
# min_prop

In [ ]:
# pattren_names_sub_df <- pattren_names_df[histone_motif_prop_with_names_df %>%
#     filter(prop >= min_prop) %>%
#     distinct(pattern) %>%
#     pull(pattern),]

In [ ]:
# nrow(pattren_names_sub_df)

In [ ]:
# head(pattren_names_sub_df)

In [ ]:
# pattren_names_sub_df <- merge(pattren_names_sub_df, manual_names_df, by = "pattern",all.x = TRUE,,sort = FALSE)


In [ ]:
# nrow(pattren_names_sub_df)

In [ ]:
tail(pattren_names_df,15)

In [ ]:
# # Find rows where Selected_TF is missing (NA)
# idx <- is.na(pattren_names_df$Selected_TF)

# # For those rows:
# # 1. Split the 'all_matches' string by ";"
# # 2. Take the first element of the split (the first TF name)
# # 3. Assign it to Selected_TF
# pattren_names_sub_df$Selected_TF[idx] <-
#   sapply(strsplit(pattren_names_sub_df$all_matches[idx], ";"), `[`, 1)


# # Find rows where pattern_num is missing (NA)
# idx <- is.na(pattren_names_sub_df$pattern_num)

# # For those rows:
# # 1. Extract the part of the string after the last "_" in the 'pattern' column
# # 2. This corresponds to the pattern number (e.g., "pos_patterns.pattern_336" → "336")
# # 3. Assign it to pattern_num
# pattren_names_sub_df$pattern_num[idx] <-
#   sub(".*_", "", pattren_names_sub_df$pattern[idx])

# pattren_names_sub_df$cluster <- NULL

In [ ]:
tail(pattren_names_df,15)

In [ ]:
head(sub_histone_scaled)

In [ ]:
unique(pattren_names_df$cluster)

In [ ]:
# sub_histone_scaled$all_matches <- sub_histone_scaled$rowname


In [ ]:
# head(sub_histone_scaled)

In [ ]:
# merged_df <-merge(pattren_names_sub_df, sub_histone_scaled[,c('all_matches','cluster')], 
#       by = "all_matches",sort = FALSE)

# # Reorder rows to match the order in sub_histone_scaled
# merged_df <- merged_df[
#   match(sub_histone_scaled$all_matches, merged_df$all_matches),
# ]

In [ ]:
# head(merged_df)

In [ ]:
# nrow(merged_df)

In [ ]:
# merged_df$to_plot <- 1

In [ ]:
# head(merged_df)

In [ ]:
# min_prop

In [ ]:
# write.csv(merged_df, "5e_EP_motifs_min_prop_0.01.csv", row.names = FALSE)

In [ ]:
# head(sub_histone_scaled)

In [ ]:
head(pattren_names_df)

In [ ]:
# nrow(sub_histone_scaled)

In [ ]:
library(ComplexHeatmap)
library(grid)

n_rows <- nrow(pattren_names_df)
row_split_vec <- pattren_names_df$cluster

make_text_heatmap <- function(labels, title, width_cm = 3, fontsize = 8) {
  
  mat <- matrix(
    1,
    nrow = length(labels),
    ncol = 1,
    dimnames = list(rownames(pattren_names_df), title)
  )
  
  Heatmap(
    mat,
    name = title,
    col = c("1" = "white"),
    
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    
    row_split = row_split_vec,
    gap = unit(1, "mm"),
    
    show_row_names = FALSE,
    show_column_names = FALSE,
    row_title = NULL,
    show_heatmap_legend = FALSE,
    
    rect_gp = gpar(col = "black", lwd = 0.5),
    border = TRUE,
    
    cell_fun = function(j, i, x, y, width, height, fill) {
      grid.text(
        labels[i],
        x = x,
        y = y,
        just = "center",
        gp = gpar(fontsize = fontsize, col = "black")
      )
    },
    
    width = unit(width_cm, "cm")
  )
}

Selected_TF_ht <- make_text_heatmap(
  labels = as.character(pattren_names_df$KC_assigned_names),
  title = "Selected_TF",
  width_cm = 4
)

pattern_num_ht <- make_text_heatmap(
  labels = as.character(pattren_names_df$pattern_num),
  title = "pattern_num",
  width_cm = 1.5
)

outfile <- "Enh_TSS_Motif_TF_pattern_table_bordered.pdf"

pdf(
  outfile,
  width = 5.5,
  height = n_rows * 0.35,
  useDingbats = FALSE
)

ht <- Selected_TF_ht + pattern_num_ht
draw(ht, gap = unit(3, "mm"))

dev.off()
message("Saved: ", outfile)

In [ ]:
outfile <- file.path(paste0("Enh_TSS_Motif_plot.pdf_with_table.pdf"))

pdf(outfile, width = 10, height = 30, useDingbats = FALSE)

ht <- Selected_TF_ht + pattern_num_ht + h2+h3

draw(
  ht,
  gap = unit(3, "mm"),
  heatmap_legend_side = "bottom"
)

dev.off()

message("Saved: ", outfile)

In [ ]:
ht <- h2+h3